In [4]:
import sys
import os

sys.path.append('/scratch2/mrenaudin/colorlessgreenRNNs')

In [5]:
from src.language_models import model as m
import torch
from utils import NounPPDataset, collate_fn_nounpp
from src.language_models.dictionary_corpus import Dictionary
from torch.utils.data import DataLoader
from collections import defaultdict



/home/mrenaudin/.conda/envs/leaps3/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
device = torch.device('cpu')
model = m.CBR_RNN(50001, 1024, 1024, 1, 0, device)
checkpoint = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/test_attention_1024/epoch_28.pt', map_location='cpu')
data_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data"
dictionary = Dictionary(data_path)
nounpp = "//scratch2/mrenaudin/colorlessgreenRNNs/NounPP/Stimuli/nounpp.txt"


In [10]:
test_dataset = NounPPDataset(nounpp, dictionary)
test_dataloader = DataLoader(test_dataset, batch_size=1024, collate_fn=collate_fn_nounpp)

In [11]:
model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [12]:
temp = checkpoint['temperature'] #that's because of mistake in save checkpoints function

In [13]:
checkpoint

{'epoch': 28,
 'model_state_dict': OrderedDict([('encoder.weight',
               tensor([[ 0.0448,  0.2151,  0.1285,  ..., -0.0111, -0.2614,  0.0806],
                       [-0.1007,  0.4339, -0.2565,  ...,  0.2145, -0.0578,  0.0452],
                       [-0.1252,  0.0740, -0.0762,  ..., -0.0687,  0.1882,  0.1473],
                       ...,
                       [-0.2784,  0.1711,  0.2999,  ..., -0.0483, -0.3118, -0.1943],
                       [-0.2245, -0.1378,  0.0398,  ...,  0.1055, -0.0432,  0.4900],
                       [-0.1109,  0.0109,  0.2937,  ...,  0.2297,  0.1861, -0.1327]])),
              ('q.weight',
               tensor([[-0.1131,  0.0332, -0.1042,  ...,  0.0583,  0.1147,  0.3531],
                       [-0.0459,  0.2877,  0.2722,  ...,  0.3172,  0.0835,  0.3865],
                       [-0.1003,  0.0791, -0.0475,  ...,  0.0974,  0.1226,  0.1297],
                       ...,
                       [-0.3688, -1.0224, -0.0519,  ...,  0.3733, -0.1918,  0.0373

In [14]:
def eval(model, test_dataloader, temperature):
    condition_accuracies = defaultdict(int)
    condition_counts = defaultdict(int)
    correct_pred = 0
    sentence_details = []
    model.eval()
    # Forward pass with hidden state update word by word
    with torch.no_grad():
        for batch in test_dataloader:
            out = None
            written = batch["sentence"]
            sentence = batch["encoded_sentence"]
            correct = batch["encoded_correct"]
            wrong = batch["encoded_wrong"]
            condition = batch["condition"]
            batch_size = sentence.size(0)

            sent = sentence[:, :5].transpose(0, 1)
            cache = model.init_cache(sent,1)  # regarder si on peut mettre du priming
            # for i in range(sent.shape[1]):
            out, cache = model(sent, cache, 1, temperature, True)
            log_probs = torch.nn.functional.log_softmax(
                out, dim=-1
            )  # s(out.squeeze(0))
            # déja sur correct et wrong log probs, pas les même résultats que sur extract_predictions.py
            correct_log_probs = log_probs[
                -1, torch.arange(batch_size), correct
            ]  # Shape: [512]
            wrong_log_probs = log_probs[-1, torch.arange(batch_size), wrong]
            correct_predictions = correct_log_probs >= wrong_log_probs

            for i in range(batch_size):
                cond = condition[i]
                pred = correct_predictions[i].item()  # Convert tensor to Python boolean
                condition_counts[cond] += 1
                condition_accuracies[cond] += pred

                sentence_details.append(
                    {
                        "sentence": written[i],
                        "condition": condition[i],
                        "correct_log_prob": correct_log_probs[i],
                        "wrong_log_prob": wrong_log_probs[i],
                        "model_prefers_correct": pred,
                    }
                )

    final_accuracies = {
        cond: condition_accuracies[cond] / condition_counts[cond]
        for cond in condition_accuracies
    }
    return final_accuracies

In [15]:
eval(model, test_dataloader, temp)

{'singular singular': 0.828,
 'singular plural': 0.571,
 'plural singular': 0.945,
 'plural plural': 0.886}